# HR Analytics – Predict Employee Attrition
## Phase 6: Model Explainability with SHAP

**Objective:** Use SHAP (SHapley Additive exPlanations) based on cooperative game theory to explain the predictions of our best-performing Random Forest model. SHAP allows us to see not only which features are globally important, but also *how* and in *which direction* they push the prediction for individual employees.

---

### 1. Setup & Load Model and Data

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import shap

warnings.filterwarnings('ignore')
shap.initjs()

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
MODEL_PATH = PROJECT_ROOT / 'models' / 'random_forest_artifact.joblib'
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'hr_ml_ready.csv'
IMAGES_DIR = PROJECT_ROOT / 'images'
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from explainability import load_model_and_data, compute_shap_values, generate_shap_plots

In [ ]:
# Load best model and data features
model, X = load_model_and_data(MODEL_PATH, DATA_PATH)
print(f"Loaded Random Forest model. Features: {X.shape[1]} | Samples: {X.shape[0]}")

### 2. Compute SHAP Values

We use `TreeExplainer` since Random Forest is an ensemble of decision trees. This computes exact SHAP values efficiently.

In [ ]:
shap_values, explainer = compute_shap_values(model, X)
print(f"Computed SHAP values successfully. Shape: {shap_values.shape}")

---
## 3. Global Explainability

### 3.1 SHAP Summary Plot

The summary plot combines feature importance with feature effects. Each point represents an employee:
- **Y-axis:** Features ordered by global importance.
- **X-axis:** SHAP value (positive values mean higher attrition probability, negative values mean lower).
- **Color:** Feature value (Red = High, Blue = Low).

In [ ]:
# Index class 1 (attrition) from the shape (samples, features, classes)
shap_exp = shap_values[:, :, 1] if len(shap_values.shape) == 3 else shap_values

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_exp, X, show=False)
plt.title('SHAP Summary Plot – Feature Attrition Impact', fontsize=14, pad=15)
plt.tight_layout()
plt.savefig(IMAGES_DIR / 'shap_summary_plot.png', dpi=120, bbox_inches='tight')
plt.show()

### 3.2 SHAP Bar Plot

The bar plot displays the mean absolute SHAP value for each feature, ranking them by average global impact.

In [ ]:
plt.figure(figsize=(10, 6))
shap.plots.bar(shap_exp, show=False)
plt.title('SHAP Feature Importance (Mean Absolute Impact)', fontsize=14, pad=15)
plt.tight_layout()
plt.savefig(IMAGES_DIR / 'shap_bar_plot.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 4. Local Explainability

### 4.1 SHAP Waterfall Plot

A waterfall plot shows how individual features pushed a specific employee's prediction away from the baseline average prediction ($E[f(x)]$) to the final prediction ($f(x)$).

In [ ]:
# Explain prediction for Employee Index 0
plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap_exp[0], show=False)
plt.title('SHAP Waterfall Plot – Employee Index 0 Prediction', fontsize=14, pad=15)
plt.tight_layout()
plt.savefig(IMAGES_DIR / 'shap_waterfall_plot.png', dpi=120, bbox_inches='tight')
plt.show()

### 4.2 SHAP Force Plot

The force plot visualizes forces that push the prediction high (red) or low (blue).

In [ ]:
expected_value = shap_exp.base_values
if hasattr(expected_value, '__len__') and len(expected_value) > 1:
    expected_value = expected_value[0]

# Create HTML visualization and save it
force_plot = shap.force_plot(
    expected_value,
    shap_exp.values[0],
    X.iloc[0],
    matplotlib=False
)
shap.save_html(str(IMAGES_DIR / 'shap_force_plot.html'), force_plot)
print(f"Saved interactive SHAP force plot HTML to {IMAGES_DIR / 'shap_force_plot.html'}")

---
## 5. HR Actionable Interpretations

Based on the SHAP values, we draw the following conclusions for HR:

1. **Most Influential Push Factors (Attrition Drivers):**
   - **OverTime_Yes:** Having overtime is the single strongest positive driver of attrition. Red dots (having overtime) push predictions significantly to the right (higher risk), indicating severe burnout risk.
   - **MonthlyIncome / JobLevel:** Lower incomes and lower job levels are heavily associated with high attrition (blue dots push to the right). This underscores that compensation dissatisfaction is a major driver.
   - **DistanceFromHome:** Higher commute distances (red dots) push predictions to the right. COMMUTE fatigue is a real retention threat.
   - **Age / Tenure_Category_Newbie (0-2):** Being young and having low company tenure are associated with voluntary exit.

2. **Most Influential Pull Factors (Retention Drivers):**
   - **StockOptionLevel:** Having stock options (especially levels 1 & 2) acts as a strong anchor, pulling the attrition risk down.
   - **YearsAtCompany / TotalWorkingYears:** Higher overall tenure and working experience decrease attrition risk (red dots push to the left).
   - **JobInvolvement / EnvironmentSatisfaction:** High involvement and high work satisfaction levels pull risk down, validating investment in workplace culture.

**Business Strategy Link:** SHAP tells us that retention isn't just about paying more. Reducing or compensating overtime, facilitating hybrid work to resolve long commutes, and structuring early-tenure mentoring are equally critical.